<a id="sim_top"></a>
# 🎲 Headhunting & Issuing Simulation Module / 終末地卡池數據模擬器

Welcome! This notebook is dedicated to executing the core statistical gacha simulations. You can click the interactive links below to jump directly to any section, or utilize the **OUTLINE** panel in VS Code for a smooth navigation experience.

歡迎使用數據模擬器！本檔案專用於執行核心的卡池數據模擬統計。您可以直接點擊下方的互動式導覽連結秒跳至指定區塊，或利用 VS Code 的 **OUTLINE（大綱）** 面板快速跳轉，閱讀體驗最佳。

---

**🇺🇸 English Version - Quick Navigation Menu**
***

* 🛠️ **[0. Dependencies & Global Parameter Settings / 基本套件匯入與全域參數設定](./simulation.ipynb#sim_mode_0)**
* 🎯 **[1. Headhunting Banner - First Acquisition of Rate-up Operator / 角色池模擬—首次獲得限定角色](./simulation.ipynb#sim_mode_1)**
* 👑 **[2. Headhunting Banner - Full Potential Rate-up Operator / 角色池模擬—滿潛限定角色](./simulation.ipynb#sim_mode_2)**
* 📊 **[3. Special Analysis - Arsenal Ticket & Arsenal Issues Conversion / 專題分析—武庫配額獲取與武器申領次數轉換](./simulation.ipynb#sim_mode_3)**
* ⚔️ **[4. Arsenal Issues - First Acquisition of Rate-up Weapon / 武器池模擬—首次獲取機率提升武器](./simulation.ipynb#sim_mode_4)**
* ⚙️ **[5. Arsenal Issues - Full Potential Rate-up Weapon / 武器池模擬—滿潛機率提升武器](./simulation.ipynb#sim_mode_5)**

[📊 Go to Visualization Module](./visualization.ipynb#vis_top)

---

**🇹🇼 繁體中文版 - 快速導覽功能選單**
***

* 🛠️ **[0. Dependencies & Global Parameter Settings / 基本套件匯入與全域參數設定](./simulation.ipynb#sim_mode_0)**
* 🎯 **[1. Headhunting Banner - First Acquisition of Rate-up Operator / 角色池模擬—首次獲得限定角色](./simulation.ipynb#sim_mode_1)**
* 👑 **[2. Headhunting Banner - Full Potential Rate-up Operator / 角色池模擬—滿潛限定角色](./simulation.ipynb#sim_mode_2)**
* 📊 **[3. Special Analysis - Arsenal Ticket & Arsenal Issues Conversion / 專題分析—武庫配額獲取與武器申領次數轉換](./simulation.ipynb#sim_mode_3)**
* ⚔️ **[4. Arsenal Issues - First Acquisition of Rate-up Weapon / 武器池模擬—首次獲取機率提升武器](./simulation.ipynb#sim_mode_4)**
* ⚙️ **[5. Arsenal Issues - Full Potential Rate-up Weapon / 武器池模擬—滿潛機率提升武器](./simulation.ipynb#sim_mode_5)**

[📊 切換至數據視覺化與統計分析模組](./visualization.ipynb#vis_top)

<a id="sim_mode_0"></a>
## 0. Dependencies & Global Parameter Settings / 基本套件匯入與全域參數設定

**🇺🇸 English Version**
***

This section is used to import required core packages and initialize global variables and environment settings.

**📊 Environment Initialization & Simulation Configurations**
* **Simulation Trials**: When running this cell, you can adjust the number of simulation iterations by modifying the `trials` variable (defaults to `1,000,000` times). Note that modifying this parameter will directly scale the execution time across all analysis pipelines, including **[3. Special Analysis - Arsenal Ticket & Arsenal Issues Conversion / 專題分析—武庫配額獲取與武器申領次數轉換](./simulation.ipynb#sim_mode_3)**.
* **Directory Auto-Creation**: The script will automatically check for and create a `sim_data` directory, which serves as the destination for exporting and storing CSV data files from each mode.

[⬆️ Back to Top](./simulation.ipynb#sim_top)

---

**🇹🇼 繁體中文版**
***

此區塊用於匯入專案所需之基礎套件，並進行全域變數與環境的初始化設定。

**📊 環境初始化與模擬參數設定**
* **模擬次數調整**：執行此區塊時，可透過變更變數 `trials` 來調整模擬次數（預設為 `1,000,000` 次）。請注意，調整此參數將直接影響後續所有模式的運算時間，特別是在跑 **[3. Special Analysis - Arsenal Ticket & Arsenal Issues Conversion / 專題分析—武庫配額獲取與武器申領次數轉換](./simulation.ipynb#sim_mode_3)** 的多節點密集運算時。
* **資料夾自動建立**：程式碼會自動檢查並新增 `sim_data` 資料夾，用作後續導出與存放各模式 CSV 數據檔案的目錄。

---

[⬆️ 返回頂端 / Back to Top](./simulation.ipynb#sim_top)

In [2]:
import numpy as np
import pandas as pd
import os

# trials = 1_000_000
trials = 1000

DATA_DIR = "sim_data"
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)


<a id="sim_mode_1"></a>
## 1. Headhunting Banner - First Acquisition of Rate-up Operator / 角色池模擬—首次獲得限定角色

**🇺🇸 English Version**
***

This mode utilizes Monte Carlo methods to simulate a player performing **Headhunting (HH)** from 0 pulls until **first acquiring the 6★ Operator on Rate up**.

**📊 Headhunting Mechanics & Pity Systems**
* **Base Appearance Rates**: 6★: `0.8%` | 5★: `8.0%` | 4★: `91.2%`.
* **Fail-Safe Mechanisms ("Pity")**:
  * **5★ Guarantee**: A 5★ (at minimum) Operator is guaranteed to appear every `10` pulls. If a 6★ and 5★ are triggered simultaneously, the 6★ Operator takes priority.
  * **6★ Rate-up Boost (Soft/Hard Pity)**: If a 6★ Operator does not appear after `65` pulls, each subsequent pull thereafter receives a `+5.0%` rate-up boost (e.g., 5.8% at the 66th pull). The 80th pull guarantees a 6★ Operator (`100%` Hard Pity). The pull count and increased rate reset as soon as any 6★ Operator appears.
  * **Rate-up (50/50)**: When obtaining a 6★ Operator, there is a `50%` chance to get the Rate-up operator (`up_6_star_char`) and a `50%` chance to get a non-Rate-up operator.
* **Banner-Exclusive Special Rules**:
  * **120-Pull One-Time Guarantee**: If the player does not get the Rate-up 6★ Operator after `119` pulls, the `120th` pull is **guaranteed** to be the 6★ Operator on Rate up. This is a one-time trigger and deactivates immediately if the Rate-up Operator is obtained early (does not carry over to future banners).
  * **30-Pull Milestone Bonus (10 Free Pulls)**: Reaching a cumulative `30` pulls grants `10` free pulls. These free pulls **do not count towards or advance** the standard pity or the 120-pull guarantee counters, but they do follow standard base probabilities and 5★ guarantee rules. Obtaining a Rate-up Operator during these 10 free pulls will **not** reset the 120-pull guarantee counter.
  * **60-Pull Milestone Voucher**: Rewards a 10-pull ticket for the next upcoming banner. It has no impact on the current banner and is **ignored** in this simulation.
  * **240-Pull Milestone**: Every `240th` pull, a Potential Token of the 6★ Operator on Rate up is awarded (counted directly into the `up_6_star_char` total).

**🔗 Sub-product Output & Conversion**
Performing Headhunting pulls actively generates the following sub-products and tokens based on the operators rolled and their duplication/potential status:
* **`Arsenal Ticket`**: Obtained from standard rolling outcomes, used for Arsenal Issue.
* **`red_quota`**: A high-value premium token rewarded when pulling duplicates of high-rarity operators.
* **`blue_quota`**: A secondary token generated from repeating lower-to-mid rarity operator acquisitions.
* For extended tracking models or budget estimation using token conversion yields, see also **[2. Headhunting Banner - Full Potential Rate-up Operator / 角色池模擬—滿潛限定角色](./simulation.ipynb#sim_mode_2)** and **[3. Special Analysis - Arsenal Ticket & Arsenal Issues Conversion / 專題分析—武庫配額獲取與武器申領次數轉換](./simulation.ipynb#sim_mode_3)**.

> 💡 **Data Note**: For exact token acquisition formulas, conversion mathematical models, and late-game full-potential environment assumptions, please refer directly to the initialization section of **[visualization.ipynb (0. Dependencies, Plot Settings & Gacha Quota Conversion)](./visualization.ipynb#vis_mode_0)**.

[⬆️ Back to Top](./simulation.ipynb#sim_top)

---

**🇹🇼 繁體中文版**
***

本模式透過蒙地卡羅方法，模擬玩家在特定角色限定池中，從 0 抽開始直到**首次獲得當期限定六星角色（UP 角色）**的消耗抽數。

**📊 核心抽卡與保底規則**
* **基礎機率**：六星 `0.8%` | 五星 `8.0%` | 四星 `91.2%`。
* **常規保底機制**：
  * **五星保底**：最多 `10` 抽必定保底獲得五星角色（若與六星同時觸發，以六星角色為優先）。
  * **六星保底（軟/硬保底）**：連續 `65` 抽未獲得六星角色時，自第 `66` 抽起，六星機率每抽依次提高 `+5.0%`，直到第 `80` 抽達到 100%（硬保底）。獲得六星角色後，下一抽機率恢復為初始的 `0.8%`。
  * **常規保底（50/50）**：獲得六星角色時，有 `50%` 機率為當期限定角色（`up_6_star_char`），`50%` 機率為非限定角色。
* **當期限定池特殊機制**：
  * **120 抽獨立大保底**：若前 `119` 抽皆未獲得限定六星，則第 `120` 抽**必定**為當期限定六星角色。若提前抽中限定六星，則此大保底機制失效，且該機制獨立計數、不會繼承至下期卡池。
  * **30 抽送 10 免費抽**：累計抽數達 30 次時觸發。此 10 抽**不計入且不推進**常規保底與 120 抽大保底進度，但享有常規機率與五星保底規則。若這 10 抽意外抽中限定角色，**不會重置**原本 120 抽的大保底計數條。
  * **60 抽送下期限定池十連抽券**：由於不影響本次限定池的抽取行為與概率，在模擬中予以**忽略**。
  * **240 抽里程碑獎勵**：每累計抽數達 `240` 次，系統額外贈送一個限定六星角色信物（計入 `up_6_star_char` 數量中）。

**🔗 副產物配額與代幣轉換**
進行尋訪抽取時，系統會根據抽出的角色稀有度及重複/潛能獲得狀況，自動產出並累加以下副產物代幣：
* **武庫配額**：卡池抽取常規產出的核心代幣，用於武庫申領。
* **紅色配額**：高價值的高階代幣，於抽中高稀有度重複角色時獲取。
* **藍色配額**：主要隨中低稀有度角色重複獲取時轉換而來。
* 關於長線累積或代幣轉換預算評估，亦可參閱 **[2. Headhunting Banner - Full Potential Rate-up Operator / 角色池模擬—滿潛限定角色](./simulation.ipynb#sim_mode_2)** 與 **[3. Special Analysis - Arsenal Ticket & Arsenal Issues Conversion / 專題分析—武庫配額獲取與武器申領次數轉換](./simulation.ipynb#sim_mode_3)**。

> 💡 **數據提示**：關於各式代幣配額的精確獲取公式、轉換數學模型與滿潛環境下的極值假設，請導向參閱 **[visualization.ipynb (0. 基礎套件、繪圖設定與角色池各種配額轉換)](./visualization.ipynb#vis_mode_0)**。

---

[⬆️ 返回頂端 / Back to Top](./simulation.ipynb#sim_top)

In [2]:
class get_one_UPcharacter:
    def __init__(self):
        # Base probability and pity parameters
        self.PROB_6_STAR = 0.008
        self.PROB_5_STAR = 0.08
        self.SOFT_PITY_START = 65
        self.HARD_PITY_6_STAR = 80
        self.HARD_PITY_5_STAR = 10
        self.BIG_PITY_LIMIT = 120
        
        # State counters (Main Banner)
        self.num_pulls_for_char = 0    # Total actual paid pulls consumed
        self.pity_6_star_count = 0     # Pulls since last 6-star character
        self.pity_5_star_count = 0     # Pulls since last 5-star character
        self.big_pity_counter = 0      # Progress towards the 120-pull big pity
        
        self.is_big_pity_active = True # 120-pull big pity, effective ONCE per Banner
        self.has_obtained_up = False   
        
        # Inventory records
        self.inventory = {
            'up_6_star_char': 0,
            'non_6_star_char': 0,
            '5_star_char': 0,
            '4_star_char': 0
        }

    def _get_current_prob_6(self):
        """Calculate soft pity probability scaling based on current pity count"""
        current_pity = self.pity_6_star_count + 1
        if current_pity <= self.SOFT_PITY_START:
            return self.PROB_6_STAR
        return self.PROB_6_STAR + (current_pity - self.SOFT_PITY_START) * 0.05

    def pull(self):
        """Execute a single paid pull"""
        self.num_pulls_for_char += 1
        if self.is_big_pity_active:
            self.big_pity_counter += 1

        prob_6 = self._get_current_prob_6()
        
        # Pity conditions
        force_up_6 = (self.is_big_pity_active and self.big_pity_counter >= self.BIG_PITY_LIMIT)
        force_any_6 = (self.pity_6_star_count + 1 >= self.HARD_PITY_6_STAR)
        force_any_5 = (self.pity_5_star_count + 1 >= self.HARD_PITY_5_STAR)

        rand = np.random.random()

        # 1. Evaluate 6-star drop
        if rand < prob_6 or force_any_6 or force_up_6:
            self.pity_6_star_count = 0
            self.pity_5_star_count = 0
            
            is_up = False
            if force_up_6:
                is_up = True
            else:
                is_up = np.random.random() < 0.5
            
            if is_up:
                self.inventory['up_6_star_char'] += 1
                self.has_obtained_up = True
                self.is_big_pity_active = False
            else:
                self.inventory['non_6_star_char'] += 1
        
        # 2. Evaluate 5-star drop (Protected from interval stacking anomaly)
        elif rand < (self.PROB_6_STAR + self.PROB_5_STAR) or force_any_5:
            self.pity_5_star_count = 0
            self.pity_6_star_count += 1
            self.inventory['5_star_char'] += 1
        
        # 3. Evaluate 4-star drop
        else:
            self.pity_6_star_count += 1
            self.pity_5_star_count += 1
            self.inventory['4_star_char'] += 1

    def pull_free(self, free_counters):
        """
        Execute a single free pull with completely isolated base probabilities.
        Receives mutable dict to persist free pity across the 10-pull loop.
        """
        Temp_PROB_6_STAR = 0.008
        Temp_PROB_5_STAR = 0.08
        Temp_HARD_PITY_5_STAR = 10

        # Increment transient counters dedicated to the free Banner
        free_counters['pity_6'] += 1
        free_counters['pity_5'] += 1

        # Check if the 5-star hard pity is triggered within the free Banner
        force_any_5 = (free_counters['pity_5'] >= Temp_HARD_PITY_5_STAR)

        rand = np.random.random()
        
        # 1. Evaluate 6-star drop (Strictly uses base probability 0.8%)
        if rand < Temp_PROB_6_STAR:
            # Reset free Banner counters upon hitting a 6-star
            free_counters['pity_6'] = 0
            free_counters['pity_5'] = 0

            Temp_is_up = np.random.random() < 0.5
            if Temp_is_up:
                self.inventory['up_6_star_char'] += 1
                self.has_obtained_up = True  # Target acquired, terminates the simulation
            else:
                self.inventory['non_6_star_char'] += 1
        
        # 2. Evaluate 5-star drop (Triggers on base 8% OR 10-pull free hard pity)
        elif rand < (Temp_PROB_6_STAR + Temp_PROB_5_STAR) or force_any_5:
            # Reset 5-star counter only for the free Banner
            free_counters['pity_5'] = 0
            self.inventory['5_star_char'] += 1
            
        # 3. Evaluate 4-star drop
        else:
            self.inventory['4_star_char'] += 1

    def run(self):
        """Simulate a single player process until obtaining at least 1 copy of UP character"""
        while not self.has_obtained_up:
            # [1] Execute standard paid pull (num_pulls_for_char increments internally)
            self.pull()
            
            # [2] Core Mechanism: Unconditionally trigger free 10-pull exactly at the 30th paid pull
            if self.num_pulls_for_char == 30:
                # Create transient counters valid only within this specific 10-pull lifecycle
                free_banner_counters = {'pity_6': 0, 'pity_5': 0}
                
                # Execute all 10 free pulls as a guaranteed bonus, appending rewards to inventory
                for _ in range(10):
                    self.pull_free(free_banner_counters)
            
            # [3] Centralized Termination Check
            # Evaluates graduation state post-paid pull or post-free 10-pull combined
            if self.has_obtained_up:
                break
        
        return {
            'num_pulls_for_char': self.num_pulls_for_char,
            'up_6_star_char': self.inventory['up_6_star_char'],
            'non_6_star_char': self.inventory['non_6_star_char'],
            '5_star_char': self.inventory['5_star_char'],
            '4_star_char': self.inventory['4_star_char']
        }

# --- Execute Simulations Pipeline ---
def main_process(total_sims):
    results = []
    print(f"Generating {total_sims} simulation records for Single UP Character...")
    
    # 💡 String-based path construction using standard os.path
    save_path = os.path.join(DATA_DIR, "sim_char_single.csv")
    
    # 💡 Core Enhancement: Safe pre-execution file purging via os module
    # If a residual leftover file exists, delete it individually to guarantee data purity
    if os.path.exists(save_path):
        os.remove(save_path) # Safely removes the single corrupted or leftover CSV
        
    # Prevent ZeroDivisionError when total_sims is set below 10 for rapid validation
    progress_step = max(1, total_sims // 10)
    
    for i in range(total_sims):
        sim = get_one_UPcharacter()
        results.append(sim.run())
        
        # Progress report evaluated with guarded step length
        if (i + 1) % progress_step == 0:
            print(f"Progress: {(i + 1) / total_sims * 100:.0f}%")
            
    df = pd.DataFrame(results)
    
    # Save to CSV
    df.to_csv(save_path, index=False)
    print(f"Done! Raw data safely exported and saved to: {save_path}\n")

if __name__ == "__main__":
    main_process(trials)

Generating 1000 simulation records for Single UP Character...
Progress: 10%
Progress: 20%
Progress: 30%
Progress: 40%
Progress: 50%
Progress: 60%
Progress: 70%
Progress: 80%
Progress: 90%
Progress: 100%
Done! Raw data safely exported and saved to: sim_data\sim_char_single.csv



<a id="sim_mode_2"></a>
## 2. Headhunting Banner - Full Potential Rate-up Operator / 角色池模擬—滿潛限定角色

**🇺🇸 English Version**
***

**🔄 Core Logic Inheritance**
This mode directly inherits the simulation environment, base appearance rates, and standard soft/hard pity counters from **[1. Headhunting Banner - First Acquisition of Rate-up Operator / 角色池模擬—首次獲得限定角色](./simulation.ipynb#sim_mode_1)**.

**🎯 Termination Condition & Focus (Core Differences)**
* **Termination Rule**: The simulation terminates only when the player successfully obtains **6 or more copies** of the Rate-up 6★ Operator (`up_6_star_char`), achieving **Full Potential**.
* **120-Pull Guarantee Deactivation**: In alignment with banner rules, the 120-pull independent guarantee is strictly a **one-time trigger**. As soon as the player obtains the **first** copy of `up_6_star_char` (either via early luck or the 120th-pull fail-safe), this mechanism deactivates (`False`) for the remainder of the session. It does not reset or provide further safety nets for subsequent copies.
* **240-Pull Milestone Accumulation**: Since the 120-pull safety net deactivates early, multi-copy progression heavily relies on this mechanic. The 240-pull counter continuously accumulates across the entire session; every 240 cumulative pulls, 1 extra Potential Token of the Rate-up Operator is awarded, serving as the primary cyclic fail-safe for full-potential targeting, completely insulating the player from absolute worst-case scenarios.

**🔗 Sub-product Output & Conversion**
* **Tracked Tokens**: Actively tracks `Arsenal Ticket`, `red_quota`, and `blue_quota` generated from standard rolling outcomes (milestones excluded).
* > 💡 **Data Note**: For exact token acquisition formulas and full-potential conversion models, please refer directly to the initialization section of **[visualization.ipynb (0. Dependencies, Plot Settings & Gacha Quota Conversion)](./visualization.ipynb#vis_mode_0)**.

[⬆️ Back to Top](./simulation.ipynb#sim_top)

---

**🇹🇼 繁體中文版**
***

**🔄 核心邏輯繼承**
本模式之底層尋訪環境、基礎抽卡機率、常規軟/硬保底計數器，完全沿用自 **[1. Headhunting Banner - First Acquisition of Rate-up Operator / 角色池模擬—首次獲得限定角色](./simulation.ipynb#sim_mode_1)**。

**🎯 終止條件與焦點（核心差異）**
* **終止判定規則**：模擬的終止條件變更為「玩家成功獲得 **6 次或以上** 的當期限定六星角色（`up_6_star_char`）」，即達到官方定義之**滿潛狀態**。
* **120 抽大保底失效判定**：依據卡池真實規則，120 抽獨立大保底為**完全的一次性機制**。玩家只要獲得**第一次（首隻）** `up_6_star_char`（不論是前 119 抽提前出貨，還是強行觸發第 120 抽保底），該保底機制即刻宣告失效（Flag 變更為 `False`）。後續抽取剩餘 5 隻的過程中，將不再享有 120 抽必定限定的保障。
* **240 抽里程碑循環累加**：由於 120 抽大保底在出貨一次後便關閉，後續長期抽取將高度依賴此機制。240 抽的里程碑計數器在整個滿潛過程中持續累計，每累計滿 240 抽便無限循環贈送一個限定六星信物，成為最壞情況下的絕對硬保底。

**🔗 副產物配額與代幣轉換**
* **統計欄位**：精準統計常規抽取產出的 `Arsenal Ticket` (武庫配額)、`red_quota` (紅色配額) 與 `blue_quota` (藍色配額)。
* > 💡 **數據提示**：關於各式代幣配額的精確獲取公式與滿潛環境下的轉換極值假設，請導向參閱 **[visualization.ipynb (0. 基礎套件、繪圖設定與角色池各種配額轉換)](./visualization.ipynb#vis_mode_0)**。

---

[⬆️ 返回頂端 / Back to Top](./simulation.ipynb#sim_top)

In [3]:
class get_Full_Potential_UPcharacter:
    def __init__(self):
        # Base probability and pity parameters
        self.PROB_6_STAR = 0.008
        self.PROB_5_STAR = 0.08
        self.SOFT_PITY_START = 65
        self.HARD_PITY_6_STAR = 80
        self.HARD_PITY_5_STAR = 10
        self.BIG_PITY_LIMIT = 120
        
        # State counters (Main Banner)
        self.num_pulls_for_char = 0    # Total actual paid pulls consumed
        self.pity_6_star_count = 0     # Pulls since last 6-star character
        self.pity_5_star_count = 0     # Pulls since last 5-star character
        self.big_pity_counter = 0      # Progress towards the 120-pull big pity
        
        self.is_big_pity_active = True # 120-pull big pity, effective ONCE per Banner lifecycle
        
        # Inventory records
        self.inventory = {
            'up_6_star_char': 0,
            'non_6_star_char': 0,
            '5_star_char': 0,
            '4_star_char': 0
        }

    def _get_current_prob_6(self):
        """Calculate soft pity probability scaling based on current pity count"""
        current_pity = self.pity_6_star_count + 1
        if current_pity <= self.SOFT_PITY_START:
            return self.PROB_6_STAR
        return self.PROB_6_STAR + (current_pity - self.SOFT_PITY_START) * 0.05

    def _should_terminate(self):
        """
        Unified Centralized Termination Criteria.
        [EASY TO MODIFY PLACE FOR FUTURE SIMULATIONS]
        """
        # Mode 1: Full Potential Termination (Target: 6 copies of UP character)
        return self.inventory['up_6_star_char'] >= 6
        
        # Mode 2: Specific Pull Limit Termination (Uncomment below to use in the future)
        # return self.num_pulls_for_char >= 120  # e.g., 120, 240, 360, ..., 1200

    def pull(self):
        """Execute a single paid pull"""
        self.num_pulls_for_char += 1
        if self.is_big_pity_active:
            self.big_pity_counter += 1

        prob_6 = self._get_current_prob_6()
        
        # Pity conditions
        force_up_6 = (self.is_big_pity_active and self.big_pity_counter >= self.BIG_PITY_LIMIT)
        force_any_6 = (self.pity_6_star_count + 1 >= self.HARD_PITY_6_STAR)
        force_any_5 = (self.pity_5_star_count + 1 >= self.HARD_PITY_5_STAR)

        rand = np.random.random()

        # 1. Evaluate 6-star drop
        if rand < prob_6 or force_any_6 or force_up_6:
            self.pity_6_star_count = 0
            self.pity_5_star_count = 0
            
            is_up = False
            if force_up_6:
                is_up = True
            else:
                is_up = np.random.random() < 0.5
            
            if is_up:
                self.inventory['up_6_star_char'] += 1
                self.is_big_pity_active = False # Big pity acquired and consumed in this Banner
            else:
                self.inventory['non_6_star_char'] += 1
        
        # 2. Evaluate 5-star drop (Protected from interval stacking anomaly)
        elif rand < (self.PROB_6_STAR + self.PROB_5_STAR) or force_any_5:
            self.pity_5_star_count = 0
            self.pity_6_star_count += 1
            self.inventory['5_star_char'] += 1
        
        # 3. Evaluate 4-star drop
        else:
            self.pity_6_star_count += 1
            self.pity_5_star_count += 1
            self.inventory['4_star_char'] += 1

    def pull_free(self, free_counters):
        """
        Execute a single free pull with completely isolated base probabilities.
        Receives mutable dict to persist free pity across the 10-pull loop.
        """
        Temp_PROB_6_STAR = 0.008
        Temp_PROB_5_STAR = 0.08
        Temp_HARD_PITY_5_STAR = 10

        # Increment transient counters dedicated to the free Banner
        free_counters['pity_6'] += 1
        free_counters['pity_5'] += 1

        # Check if the 5-star hard pity is triggered within the free Banner
        force_any_5 = (free_counters['pity_5'] >= Temp_HARD_PITY_5_STAR)

        rand = np.random.random()
        
        # 1. Evaluate 6-star drop (Strictly uses base probability 0.8%)
        if rand < Temp_PROB_6_STAR:
            free_counters['pity_6'] = 0
            free_counters['pity_5'] = 0

            Temp_is_up = np.random.random() < 0.5
            if Temp_is_up:
                self.inventory['up_6_star_char'] += 1
            else:
                self.inventory['non_6_star_char'] += 1
        
        # 2. Evaluate 5-star drop (Triggers on base 8% OR 10-pull free hard pity)
        elif rand < (Temp_PROB_6_STAR + Temp_PROB_5_STAR) or force_any_5:
            free_counters['pity_5'] = 0
            self.inventory['5_star_char'] += 1
            
        # 3. Evaluate 4-star drop
        else:
            self.inventory['4_star_char'] += 1

    def run(self):
        """Simulate a player process until centralized termination criteria is met"""
        TOKEN_MILESTONES = {240, 480, 720, 960, 1200}
        
        while not self._should_terminate():
            # [1] Execute standard paid pull (num_pulls_for_char increments internally)
            self.pull()
            
            # [2] Core Mechanism: Unconditionally trigger free 10-pull exactly at the 30th paid pull
            if self.num_pulls_for_char == 30:
                # Create transient counters valid only within this specific 10-pull lifecycle
                free_banner_counters = {'pity_6': 0, 'pity_5': 0}
                
                # Execute all 10 free pulls as a guaranteed bonus, appending rewards to inventory
                for _ in range(10):
                    self.pull_free(free_banner_counters)
                    
            # [3] Milestone Mechanism: Distribute token rewards right after milestone paid pulls
            elif self.num_pulls_for_char in TOKEN_MILESTONES:
                self.inventory['up_6_star_char'] += 1
            
            # [4] Centralized Termination Check
            # Evaluates full graduation state post-paid, post-free, or post-token reward combined
            if self._should_terminate():
                break
        
        return {
            'num_pulls_for_char': self.num_pulls_for_char,
            'up_6_star_char': self.inventory['up_6_star_char'],
            'non_6_star_char': self.inventory['non_6_star_char'],
            '5_star_char': self.inventory['5_star_char'],
            '4_star_char': self.inventory['4_star_char']
        }

# --- Execute Simulations Pipeline ---
def main_process(total_sims):
    results = []
    print(f"Generating {total_sims} simulation records for Full Potential UP Character...")
    
    # 💡 String-based path construction using standard os.path
    save_path = os.path.join(DATA_DIR, "sim_char_full_potential.csv")
    
    # 💡 Core Enhancement: Safe pre-execution file purging via os module
    # If a residual leftover file exists, delete it individually to guarantee data purity
    if os.path.exists(save_path):
        os.remove(save_path)
        
    # Prevent ZeroDivisionError when total_sims is set below 10 for rapid validation
    progress_step = max(1, total_sims // 10)
    
    for i in range(total_sims):
        sim = get_Full_Potential_UPcharacter()
        results.append(sim.run())
        
        # Progress report evaluated with guarded step length
        if (i + 1) % progress_step == 0:
            print(f"Progress: {(i + 1) / total_sims * 100:.0f}%")
            
    df = pd.DataFrame(results)
    
    # Save to CSV
    df.to_csv(save_path, index=False)
    print(f"Done! Raw data safely exported and saved to: {save_path}\n")

if __name__ == "__main__":
    main_process(trials)

Generating 1000 simulation records for Full Potential UP Character...
Progress: 10%
Progress: 20%
Progress: 30%
Progress: 40%
Progress: 50%
Progress: 60%
Progress: 70%
Progress: 80%
Progress: 90%
Progress: 100%
Done! Raw data safely exported and saved to: sim_data\sim_char_full_potential.csv



<a id="sim_mode_3"></a>
## 3. Special Analysis - Arsenal Ticket & Arsenal Issues Conversion / 專題分析—武庫配額獲取與武器申領次數轉換

**🇺🇸 English Version**
***

**🔄 Core Logic Inheritance**
This mode directly inherits the simulation environment, base appearance rates, and gacha pity behaviors from **[1. Headhunting Banner - First Acquisition of Rate-up Operator / 角色池模擬—首次獲得限定角色](./simulation.ipynb#sim_mode_1)**. Rather than targeting a specific character acquisition goal, this mode serves as a budget bridging analysis between Headhunting and Arsenal Issues, anchoring the player's pulling equity before spending resources.

**🎯 Custom Focus & Variable Nodes (Core Differences)**
* **Fixed Pull Array (`pull_nodes`)**: The simulation evaluates cumulative token yields across a configurable array of fixed pulling thresholds. By default, it simulates 6 core benchmark nodes: `pull_nodes = [80, 120, 240, 480, 720, 960]`. Users can freely modify this array to inspect their own target budgets.
* **Syncing for Visualization**: To render the statistical distributions correctly, ensure that the `pull_nodes` array configured here perfectly matches the one defined in **[visualization.ipynb (3. Special Analysis - Arsenal Ticket & Arsenal Issues Conversion Charts)](./visualization.ipynb#vis_mode_3)**.
* **Execution Performance Tip**: If you add higher pulling nodes or want to speed up execution time, remember to navigate back to **[0. Dependencies & Global Parameter Settings / 基本套件匯入與全域參數設定](./simulation.ipynb#sim_mode_0)** and decrease the `trials` variable (defaults to `1,000,000` iterations) to avoid long waiting periods.
* **Overflow Redirect Logic (Full-Potential Cap)**: Simulating exceptionally high fixed pull counts may result in the player rolling more than 6 copies of the Rate-up Operator. To maintain mathematical precision for duplicate token conversions, any `up_6_star_char` count exceeding `6` is automatically redirected and added to the `non_up_6_star_char` variable, which can be practically interpreted as pulling a non-Rate-up Operator.

**🔗 Sub-product Output & Conversion**
* **Tracked Tokens**: Focuses primarily on compiling the total accumulated `Arsenal Ticket` yields across different pulling milestones, along with companion `red_quota` and `blue_quota` metrics.
* > 💡 **Data Note**: For exact token acquisition formulas and conversion mathematical models, please refer directly to the initialization section of **[visualization.ipynb (0. Dependencies, Plot Settings & Gacha Quota Conversion)](./visualization.ipynb#vis_mode_0)**.

[⬆️ Back to Top](./simulation.ipynb#sim_top)

---

**🇹🇼 繁體中文版**
***

**🔄 核心邏輯繼承**
本模式之底層尋訪環境、基礎抽卡機率與保底行為，完全沿用自 **[1. Headhunting Banner - First Acquisition of Rate-up Operator / 角色池模擬—首次獲得限定角色](./simulation.ipynb#sim_mode_1)**。本模式不以特定角色獲取為終止條件，而是作為玩家從「角色尋訪」跨入「武庫申領」的預算過渡橋樑，旨在幫助玩家在實際投入資源前精準定錨預算期望值。

**🎯 專題焦點與節點變數（核心差異）**
* **固定抽數陣列 (`pull_nodes`)**：此處透過設定特定抽數節點來觀察副產物的累積狀況，預設模擬 6 個核心指標節點：`pull_nodes = [80, 120, 240, 480, 720, 960]`。使用者可依自身預算規劃自由修改此陣列，以觀測特定抽數下的數據。
* **跨檔案變數對齊**：在執行完模擬後，請前往 **[visualization.ipynb (3. 武庫配額獲取與武器申領次數轉換模擬結果之統計與視覺化圖表)](./visualization.ipynb#vis_mode_3)**，將兩邊的 `pull_nodes` 陣列變數保持一致，即可順利渲染出對應的統計視覺化圖表。
* **執行效能提示**：若使用者自行定義了更長或更多的抽數節點，考慮到模擬數據的運算時間，建議可回頭至 **[0. Dependencies & Global Parameter Settings / 基本套件匯入與全域參數設定](./simulation.ipynb#sim_mode_0)** 將 `trials` 模擬次數變數（預設為一百萬次）調低，以避免過長時間的黑屏等待。
* **溢出轉換邏輯（滿潛截斷）**：由於模擬較高的固定抽數可能會導致獲得的限定六星數量超過滿潛所需。為了在輸出 CSV 時維持代幣轉換的嚴謹性，當 `up_6_star_char` 超過 `6` 隻時，程式碼會將多餘的數量自動轉移至 `non_up_6_star_char` 變數中；低於 6 隻則維持原樣。在實際抽取意義中，這可完全等效為歪了非限定角色（非 UP 角色）。

**🔗 副產物配額與代幣轉換**
* **統計焦點**：專注於統計在各個固定抽數里程碑下，玩家常規累積產出的 `Arsenal Ticket` (武庫配額) 總量，並伴隨統計 `red_quota` (紅色配額) 與 `blue_quota` (藍色配額)。
* > 💡 **數據提示**：關於各式代幣配額的精確獲取公式與轉換數學模型，請導向參閱 **[visualization.ipynb (0. 基礎套件、繪圖設定與角色池各種配額轉換)](./visualization.ipynb#vis_mode_0)**。

---

[⬆️ 返回頂端 / Back to Top](./simulation.ipynb#sim_top)

In [4]:
class Fixed_Pulls:
    def __init__(self, target_pulls):
        # Base probability and pity parameters
        self.PROB_6_STAR = 0.008
        self.PROB_5_STAR = 0.08
        self.SOFT_PITY_START = 65
        self.HARD_PITY_6_STAR = 80
        self.HARD_PITY_5_STAR = 10
        self.BIG_PITY_LIMIT = 120
        
        # Target limit for this specific simulation node
        self.target_pulls = target_pulls
        
        # State counters (Main Banner)
        self.num_pulls_for_char = 0    # Total actual paid pulls consumed
        self.pity_6_star_count = 0     # Pulls since last 6-star character
        self.pity_5_star_count = 0     # Pulls since last 5-star character
        self.big_pity_counter = 0      # Progress towards the 120-pull big pity
        
        self.is_big_pity_active = True # 120-pull big pity, effective ONCE per Banner lifecycle
        
        # Inventory records
        self.inventory = {
            'up_6_star_char': 0,
            'non_6_star_char': 0,
            '5_star_char': 0,
            '4_star_char': 0
        }

    def _get_current_prob_6(self):
        """Calculate soft pity probability scaling based on current pity count"""
        current_pity = self.pity_6_star_count + 1
        if current_pity <= self.SOFT_PITY_START:
            return self.PROB_6_STAR
        return self.PROB_6_STAR + (current_pity - self.SOFT_PITY_START) * 0.05

    def _should_terminate(self):
        """
        Unified Centralized Termination Criteria.
        Mode 2: Specific Pull Limit Termination
        """
        return self.num_pulls_for_char >= self.target_pulls

    def pull(self):
        """Execute a single paid pull"""
        self.num_pulls_for_char += 1
        if self.is_big_pity_active:
            self.big_pity_counter += 1

        prob_6 = self._get_current_prob_6()
        
        # Pity conditions
        force_up_6 = (self.is_big_pity_active and self.big_pity_counter >= self.BIG_PITY_LIMIT)
        force_any_6 = (self.pity_6_star_count + 1 >= self.HARD_PITY_6_STAR)
        force_any_5 = (self.pity_5_star_count + 1 >= self.HARD_PITY_5_STAR)

        rand = np.random.random()

        # 1. Evaluate 6-star drop
        if rand < prob_6 or force_any_6 or force_up_6:
            self.pity_6_star_count = 0
            self.pity_5_star_count = 0
            
            is_up = False
            if force_up_6:
                is_up = True
            else:
                is_up = np.random.random() < 0.5
            
            if is_up:
                self.inventory['up_6_star_char'] += 1
                self.is_big_pity_active = False # Big pity acquired and consumed in this Banner
            else:
                self.inventory['non_6_star_char'] += 1
        
        # 2. Evaluate 5-star drop (Protected from interval stacking anomaly)
        elif rand < (self.PROB_6_STAR + self.PROB_5_STAR) or force_any_5:
            self.pity_5_star_count = 0
            self.pity_6_star_count += 1
            self.inventory['5_star_char'] += 1
        
        # 3. Evaluate 4-star drop
        else:
            self.pity_6_star_count += 1
            self.pity_5_star_count += 1
            self.inventory['4_star_char'] += 1

    def pull_free(self, free_counters):
        """
        Execute a single free pull with completely isolated base probabilities.
        """
        Temp_PROB_6_STAR = 0.008
        Temp_PROB_5_STAR = 0.08
        Temp_HARD_PITY_5_STAR = 10

        free_counters['pity_6'] += 1
        free_counters['pity_5'] += 1

        force_any_5 = (free_counters['pity_5'] >= Temp_HARD_PITY_5_STAR)

        rand = np.random.random()
        
        # 1. Evaluate 6-star drop (Strictly uses base probability 0.8%)
        if rand < Temp_PROB_6_STAR:
            free_counters['pity_6'] = 0
            free_counters['pity_5'] = 0

            Temp_is_up = np.random.random() < 0.5
            if Temp_is_up:
                self.inventory['up_6_star_char'] += 1
            else:
                self.inventory['non_6_star_char'] += 1
        
        # 2. Evaluate 5-star drop (Triggers on base 8% OR 10-pull free hard pity)
        elif rand < (Temp_PROB_6_STAR + Temp_PROB_5_STAR) or force_any_5:
            free_counters['pity_5'] = 0
            self.inventory['5_star_char'] += 1
            
        # 3. Evaluate 4-star drop
        else:
            self.inventory['4_star_char'] += 1

    def run(self):
        """Simulate a player process until target pull budget is fully spent"""
        TOKEN_MILESTONES = {240, 480, 720, 960, 1200}
        
        while not self._should_terminate():
            # [1] Execute standard paid pull (num_pulls_for_char increments internally)
            self.pull()
            
            # [2] Core Mechanism: Unconditionally trigger free 10-pull exactly at the 30th paid pull
            if self.num_pulls_for_char == 30:
                # Create transient counters valid only within this specific 10-pull lifecycle
                free_banner_counters = {'pity_6': 0, 'pity_5': 0}
                
                # Execute all 10 free pulls as a guaranteed bonus, appending rewards to inventory
                for _ in range(10):
                    self.pull_free(free_banner_counters)
            
            # [3] Milestone Mechanism: Distribute token rewards right after milestone paid pulls
            elif self.num_pulls_for_char in TOKEN_MILESTONES:
                self.inventory['up_6_star_char'] += 1
            
            # [4] Centralized Termination Check
            # Safely terminates the loop once the designated budget (target_pulls) is fully consumed
            if self._should_terminate():
                break
        
        return {
            'num_pulls_for_char': self.num_pulls_for_char,
            'up_6_star_char': self.inventory['up_6_star_char'],
            'non_6_star_char': self.inventory['non_6_star_char'],
            '5_star_char': self.inventory['5_star_char'],
            '4_star_char': self.inventory['4_star_char']
        }

# --- Execute Simulations Loop for All Budget Nodes ---
def main_process(total_sims):
    pull_nodes = [80, 120, 240, 480, 720, 960]

    for nodes in pull_nodes:
        results = []
        print(f"Generating {total_sims} simulation records for Fixed Pulls: {nodes}...")

        # 💡 String-based path construction using standard os.path for individual nodes
        save_path = os.path.join(DATA_DIR, f"sim_fixed_{nodes}_pulls.csv")

        # 💡 Core Enhancement: Safe pre-execution file purging for current node
        # Removes ONLY the active tracking node's file to avoid cross-mode data loss
        if os.path.exists(save_path):
            os.remove(save_path)

        # Prevent ZeroDivisionError outside the internal loop when total_sims < 10
        progress_step = max(1, total_sims // 10)

        for i in range(total_sims):
            sim = Fixed_Pulls(target_pulls=nodes)
            results.append(sim.run())

            # Progress report evaluated with guarded step length
            if (i + 1) % progress_step == 0:
                print(
                    f"Node {nodes} Pulls - Progress: {(i + 1) / total_sims * 100:.0f}%"
                )

        df = pd.DataFrame(results)

        # --- Data Shaping and Redistribution Logic (Executed prior to CSV export) ---
        # Calculate the overflow amount of UP characters (values below 6 will precisely capture as 0)
        nums_exceed_6 = (df["up_6_star_char"] - 6).clip(lower=0)
        # Transfer the overflowed copies to the standard non-up Banner category
        df["non_6_star_char"] += nums_exceed_6
        # Cap the standard UP character inventory at a maximum of 6 copies (Full Potential)
        df["up_6_star_char"] = df["up_6_star_char"].clip(upper=6)

        # Save to CSV
        df.to_csv(save_path, index=False)
        print(f"Done! Data for {nodes} pulls safely saved to: {save_path}\n")

if __name__ == "__main__":
    main_process(trials)

Generating 1000 simulation records for Fixed Pulls: 80...
Node 80 Pulls - Progress: 10%
Node 80 Pulls - Progress: 20%
Node 80 Pulls - Progress: 30%
Node 80 Pulls - Progress: 40%
Node 80 Pulls - Progress: 50%
Node 80 Pulls - Progress: 60%
Node 80 Pulls - Progress: 70%
Node 80 Pulls - Progress: 80%
Node 80 Pulls - Progress: 90%
Node 80 Pulls - Progress: 100%
Done! Data for 80 pulls safely saved to: sim_data\sim_fixed_80_pulls.csv

Generating 1000 simulation records for Fixed Pulls: 120...
Node 120 Pulls - Progress: 10%
Node 120 Pulls - Progress: 20%
Node 120 Pulls - Progress: 30%
Node 120 Pulls - Progress: 40%
Node 120 Pulls - Progress: 50%
Node 120 Pulls - Progress: 60%
Node 120 Pulls - Progress: 70%
Node 120 Pulls - Progress: 80%
Node 120 Pulls - Progress: 90%
Node 120 Pulls - Progress: 100%
Done! Data for 120 pulls safely saved to: sim_data\sim_fixed_120_pulls.csv

Generating 1000 simulation records for Fixed Pulls: 240...
Node 240 Pulls - Progress: 10%
Node 240 Pulls - Progress: 20%


<a id="sim_mode_4"></a>
## 4. Arsenal Issues - First Acquisition of Rate-up Weapon / 武器池模擬—首次獲取機率提升武器

**🇺🇸 English Version**
***

**🔄 Core Logic & Environment Settings**
Unlike Headhunting, this mode operates on a per-issue basis, where each Issue attempt consists of 10 random weapon drops and costs `1980 Arsenal Tickets`. The simulation initializes a dedicated weapon-pulling environment based on the baseline appearance rates:
* **6★ Weapon**: `4.0000%`
* **5★ Weapon**: `15.0000%`
* **4★ Weapon**: `81.0000%`

**🎯 Termination Condition**
* **Termination Rule**: The simulation terminates immediately when the player successfully obtains their **first copy** of the designated Rate-up 6★ Weapon (`up_6_star_weapon`), whether via standard random drops or the consecutive issue fail-safe system.

**🛡️ Pity & Fail-safe Mechanisms**
* **5★ or Better Guarantee Per Issue**: At least one 5★ or better weapon is guaranteed to appear in any of the issues. If the first 9 weapons in a single issue are not 5★ or 6★ weapons, then the 10th weapon will be 5★ at minimum, satisfying the safety baseline without shifting standard high-rarity drop rates.
* **3 Consecutive Issues 6★ Fail-safe**: If no 6★ weapon is obtained within 3 consecutive issues, the next (4th) issue will guarantee a 6★ weapon. This counter resets upon drawing any 6★ weapon and does not carry over to other banners.
* **Rate-up Drop Rate Split**: When obtaining a 6★ weapon, there is a fixed `25.0000%` chance to get the Rate-up 6★ Weapon (`up_6_star_weapon`). The remaining `75.0000%` is distributed equally among non-Rate-up 6★ weapons (`non_up_6_star_weapon`).
* **7 Consecutive Issues Rate-up Guarantee (One-time)**: There is a guarantee system that if no Rate-up 6★ weapon is obtained after 7 consecutive issues, the Rate-up 6★ weapon is guaranteed in the 8th issue. This safety net deactivates for the rest of the banner session once the first `up_6_star_weapon` is acquired.
* **Fail-safe Priority Control**: If the 5★ at minimum threshold and a 6★ guarantee are triggered simultaneously, the **6★ weapon takes strict priority over the 5★ weapon** to prevent programmatic interference with the guaranteed rules.

**🎁 Cumulative Milestone Reward Nodes**
The script tracks total cumulative issues to precisely trigger the milestone reward cycle. These rewards are directly injected into the weapon variables and count toward progression:
* **10th Issue**: Rewards 1 Weapon Selection Box, which is counted as `non_up_6_star_weapon`.
* **18th Issue**: Rewards 1 copy of the Rate-up 6★ Weapon (`up_6_star_weapon`) instead.
* **Subsequent Cycles (Every 8 Issues)**: The cycle follows continuously on every 8 issues afterwards, alternating between a Weapon Selection Box and a Rate-up 6★ weapon. Effectively, this guarantees a direct `up_6_star_weapon` reward at the **18, 34, 50, 66, and 82** cumulative issue marks. Note that since this mode terminates immediately upon the first acquisition, later milestones are primarily relevant for long-term progression analysis such as **[5. Arsenal Issues - Full Potential Rate-up Weapon / 武器池模擬—滿潛機率提升武器](./simulation.ipynb#sim_mode_5)**.

**🔗 Sub-product Output & Yields**
* **Tracked Metrics**: Outputs total cumulative pulls (`num_pulls_for_weapon`), `up_6_star_weapon`, `non_up_6_star_weapon`, `5_star_weapon`, and `4_star_weapon`.
* **Token Yields (`blue_quota`)**: Dynamically tallies `blue_quota` generated during the session: each obtained 6★ weapon rewards `50 blue_quota`, and each 5★ weapon rewards `10 blue_quota`.

[⬆️ Back to Top](./simulation.ipynb#sim_top)

---

**🇹🇼 繁體中文版**
***

**🔄 核心邏輯與環境設定**
本模式與角色池不同，是以**「單次申領（Issue，即一組十連抽）」**為核心操作單位，每次申領消耗 `1980 武庫配額`。模擬器將初始化獨立的武器池抽取環境，其基礎掉落機率設定如下：
* **六星武器 (6★)**：`4.0000%`
* **五星武器 (5★)**：`15.0000%`
* **四星武器 (4★)**：`81.0000%`

**🎯 終止條件**
* **終止判定規則**：當玩家透過常規隨機抽取、7 次連續申領之保底機制（Fail-safe）成功獲得**第一把（首獲）**機率提升六星武器（`up_6_star_weapon`）時，模擬即刻終止。

**🛡️ 保底與避非機制（Fail-safe）**
* **單次申領五星或以上保底**：每次申領（10抽）必定包含至少一把五星或以上（5★ or better）的武器。若前 9 抽皆未獲得 5★ 或 6★ 武器，則第 10 抽將強制觸發保底，確保該抽結果至少為五星（5★ at minimum），以符合機制避非基準。
* **連續 3 次申領六星保底**：若連續 3 次申領皆未獲得六星武器，則下一次（第 4 次）申領必定保底獲得六星武器。此計數器在抽中任何六星武器時重置，且不繼承至其他武器池。
* **機率提升拆分 (Rate-up)**：當抽中六星武器時，有固定 `25.0000%` 的機率為當期機率提升六星武器（`up_6_star_weapon`），其餘 `75.0000%` 機率均等分配給非機率提升六星武器（`non_up_6_star_weapon`）。
* **連續 7 次申領機率提升大保底（僅限一次）**：本池存在一項大保底機制：若連續 7 次申領皆未獲得機率提升六星武器，則第 8 次申領必定獲得 `up_6_star_weapon`。若提前抽出機率提升六星武器，則該機制在本次卡池期間直接失效。
* **保底優先度控制**：若同時觸發五星保底與六星保底，系統判定上 **六星優先度高於五星**，以避免程式執行時發生規則衝突。

**🎁 累計申領里程碑節點**
程式碼精準追蹤累計申領次數，用以觸發設定的里程碑獎累積循環：
* **累計 10 次申領**：獲得「非機率提升武器自選箱」，直接計入 `non_up_6_star_weapon`。
* **累計 18 次申領**：改為獲得一把當期機率提升六星武器（`up_6_star_weapon`）。
* **後續循環（每 8 次交替）**：在此之後，每申領 8 次即可額外交替獲得自選箱與機率提升武器。在實務運算中，相當於玩家在累積申領達 **18、34、50、66、82** 次等保底節點時，會直接獲得 `up_6_star_weapon`。由於本模式在「首獲」時即終止，後續更深的里程碑循環主要於 **[5. Arsenal Issues - Full Potential Rate-up Weapon / 武器池模擬—滿潛機率提升武器](./simulation.ipynb#sim_mode_5)** 中發揮長線統計作用。

**🔗 副產物配額與代幣統計**
* **統計欄位**：輸出總抽取次數 (`num_pulls_for_weapon`)、`up_6_star_weapon`、`non_up_6_star_weapon`，`5_star_weapon` 與 `4_star_weapon`。
* **代幣產出 (`blue_quota`)**：動態統計抽取過程中累積獲得的「集成配額（AIC Quota）」。每獲得一個六星武器產出 `50 blue_quota`；每獲得一個五星武器產出 `10 blue_quota`。

---

[⬆️ 返回頂端 / Back to Top](./simulation.ipynb#sim_top)

In [5]:
class get_one_UPweapon:
    def __init__(self):
        # Base probabilities for a single item inside a multi-pull
        self.PROB_6_STAR = 0.04
        self.PROB_5_STAR = 0.15
        
        # State counters
        self.num_pulls_for_weapon = 0       # Total number of "multi-pulls" consumed (1, 2, 3...)
        self.pity_6_star_singles = 0        # CRITICAL: Relative single pulls elapsed since the last 6-star
        
        # Flag for the unique 8th multi-pull guarantee
        self.is_8th_multi_pity_active = True 
        self.blue_quota = 0
        
        self.inventory = {
            'up_6_star_weapon': 0,
            'non_up_6_star_weapon': 0,
            '5_star_weapon': 0,
            '4_star_weapon': 0
        }

    def _should_terminate(self):
        # First Copy Graduation (Terminates immediately when 1st UP is obtained)
        return self.inventory['up_6_star_weapon'] >= 1

    def _roll_six_star(self, is_forced_up=False):
        """Helper to process a 6-star acquisition from pulls (Awards 50 blue_quota)"""
        self.blue_quota += 50  
        # Whichever way a 6-star is obtained, the relative counter resets to 0 immediately!
        self.pity_6_star_singles = 0 
        
        is_up = is_forced_up or (np.random.random() < 0.25)
        if is_up:
            self.inventory['up_6_star_weapon'] += 1
            self.is_8th_multi_pity_active = False # Deactivate big pity forever if UP is obtained
        else:
            self.inventory['non_up_6_star_weapon'] += 1

    def execute_one_multi_pull(self):
        """Execute exactly ONE multi-pull batch (10 slots) with real-time rolling counters"""
        self.num_pulls_for_weapon += 1
        
        # Local trackers to monitor drops within this specific multi-pull batch
        six_stars_in_this_multi = 0
        five_stars_in_this_multi = 0
        
        # Process a 10-slot loop where each slot represents a real-time single pull sequence
        for slot in range(10):
            is_last_slot = (slot == 9)
            
            # --- Establish Real-Time Pity Conditions ---
            # 1. Big Pity: Absolute 80th pull (Last slot of 8th multi)
            force_up_6 = is_last_slot and self.is_8th_multi_pity_active and (self.num_pulls_for_weapon == 8)
            
            # 2. Regular Pity: Real-time 40th rolling single pull without any 6-star weapon
            force_any_6 = (self.pity_6_star_singles + 1 == 40)
            
            # Status flag to check if the current single pull yields a 6-star
            slot_yielded_6_star = False
            
            # --- Core Priority Interception (Evaluated dynamically per single pull) ---
            if force_up_6:
                self._roll_six_star(is_forced_up=True)
                six_stars_in_this_multi += 1
                slot_yielded_6_star = True
                
            elif force_any_6:
                # Real-time regular pity intercept: Triggers immediately at the exact 40th pull
                self._roll_six_star(is_forced_up=False)
                six_stars_in_this_multi += 1
                slot_yielded_6_star = True
                
            else:
                # Standard Probability Roll for the current slot
                rand = np.random.random()
                
                # A: Check if 6-star is rolled naturally (4% chance)
                if rand < self.PROB_6_STAR:
                    self._roll_six_star(is_forced_up=False)
                    six_stars_in_this_multi += 1
                    slot_yielded_6_star = True
                
                # B: If 6-star is NOT rolled, check if 5-star is rolled naturally OR forced by 10-pull pity
                # Pity condition: Last slot AND no 5-star or 6-star was found in the previous 9 slots
                elif (rand < (self.PROB_6_STAR + self.PROB_5_STAR)) or \
                     (is_last_slot and (six_stars_in_this_multi == 0) and (five_stars_in_this_multi == 0)):
                    self.inventory['5_star_weapon'] += 1
                    self.blue_quota += 10
                    five_stars_in_this_multi += 1
                
                # C: Otherwise, it's a 4-star weapon
                else:
                    self.inventory['4_star_weapon'] += 1
            
            # --- Real-Time Counter Maintenance ---
            # Increment the rolling counter only if NO 6-star weapon was obtained in this specific single pull.
            if not slot_yielded_6_star:
                self.pity_6_star_singles += 1

    def run(self):
        """Simulate the weapon banner process per multi-pull unit until graduation"""
        while not self._should_terminate():
            # Step 1: Execute the standard multi-pull
            self.execute_one_multi_pull()
            
            # Step 2: Progress Bar Check - Directly gift items upon reaching specific milestones
            current_count = self.num_pulls_for_weapon
            
            # Pattern A: Non-up Choice Box at 10, 26, 42, 58, 74...
            if current_count == 10 or (current_count > 10 and (current_count - 10) % 16 == 0):
                self.inventory['non_up_6_star_weapon'] += 1
                
            # Pattern B: Up Weapon Milestone at 18, 34, 50, 66, 82...
            if current_count == 18 or (current_count > 18 and (current_count - 18) % 16 == 0):
                self.inventory['up_6_star_weapon'] += 1
                
            # Step 3: Centralized check immediately after all drops and gifts are resolved
            if self._should_terminate():
                break
            
        return {
            'num_pulls_for_weapon': self.num_pulls_for_weapon,
            'up_6_star_weapon': self.inventory['up_6_star_weapon'],
            'non_up_6_star_weapon': self.inventory['non_up_6_star_weapon'],
            '5_star_weapon': self.inventory['5_star_weapon'],
            '4_star_weapon': self.inventory['4_star_weapon'],
            'blue_quota': self.blue_quota
        }
    
# --- Execute Simulations Pipeline ---
def main_process(total_sims):
    results = []
    print(f"Generating {total_sims} simulation records for single UP Weapon...")

    # 💡 String-based path construction using standard os.path
    save_path = os.path.join(DATA_DIR, "sim_weapon_single.csv")

    # 💡 Core Enhancement: Safe pre-execution file purging via os module
    # If a residual leftover file exists, delete it individually to guarantee data purity
    if os.path.exists(save_path):
        os.remove(save_path)

    # Prevent ZeroDivisionError when total_sims is set below 10 for rapid validation
    progress_step = max(1, total_sims // 10)

    for i in range(total_sims):
        sim = get_one_UPweapon()
        results.append(sim.run())

        # Progress report evaluated with guarded step length
        if (i + 1) % progress_step == 0:
            print(f"Progress: {(i + 1) / total_sims * 100:.0f}%")

    df = pd.DataFrame(results)

    # Save to CSV
    df.to_csv(save_path, index=False)
    print(f"Done! Raw data safely exported and saved to: {save_path}\n")

if __name__ == "__main__":
    main_process(trials)


Generating 1000 simulation records for single UP Weapon...
Progress: 10%
Progress: 20%
Progress: 30%
Progress: 40%
Progress: 50%
Progress: 60%
Progress: 70%
Progress: 80%
Progress: 90%
Progress: 100%
Done! Raw data safely exported and saved to: sim_data\sim_weapon_single.csv



<a id="sim_mode_5"></a>
## 5. Arsenal Issues - Full Potential Rate-up Weapon / 武器池模擬—滿潛機率提升武器

**🇺🇸 English Version**
***

**🔄 Core Logic & Environment Settings**
This mode shares the exact same operational unit and gacha environment settings as **[4. Arsenal Issues - First Acquisition of Rate-up Weapon / 武器池模擬—首次獲取機率提升武器](./simulation.ipynb#sim_mode_4)**. Each Issue attempt costs `1980 Arsenal Tickets` and delivers 10 weapon drops based on the static baseline appearance rates (6★: `4.0000%`, 5★: `15.0000%`, 4★: `81.0000%`).

**🎯 Termination Condition**
* **Termination Rule**: The simulation terminates strictly when the player accumulates **6 or more copies** of the designated Rate-up 6★ Weapon (`up_6_star_weapon >= 6`), successfully reaching Full Potential.

**🛡️ Pity & Fail-safe Mechanisms**
* **5★ or Better Guarantee Per Issue**: If the first 9 weapons in a single issue are not 5★ or 6★ weapons, the 10th weapon is guaranteed to be 5★ at minimum, protecting the player without altering base high-rarity rates.
* **3 Consecutive Issues 6★ Fail-safe**: Guarantees a 6★ weapon on the 4th issue if none were drawn in the previous 3 issues. This counter resets upon drawing any 6★ weapon.
* **7 Consecutive Issues Rate-up Guarantee (One-time Activation)**: If no Rate-up 6★ weapon is obtained after 7 consecutive issues, the Rate-up 6★ weapon is guaranteed on the 8th issue. 
  * > ⚠️ **Structural Behavior**: This specific safety net triggers **at most once per banner session**. Once the player obtains their first `up_6_star_weapon` (either via luck or this 8th-issue guarantee), this fail-safe deactivates for the remainder of the simulation run.
* **Fail-safe Priority Control**: If the 5★ at minimum threshold and a 6★ guarantee trigger simultaneously, the 6★ weapon takes strict priority.

**🎁 Cumulative Milestone Reward Nodes (Core Progression Drivers)**
Because reaching Full Potential requires an extended pulling session, the script continuously tracks total cumulative issues to trigger the milestone cycle. Unlike the one-time fail-safe, these milestones act as guaranteed safety nets throughout the entire run:
* **10th Issue Milestone**: Rewards 1 Weapon Selection Box, which is programmatically added to `non_up_6_star_weapon`.
* **18th Issue Milestone**: Rewards 1 copy of the Rate-up 6★ Weapon (`up_6_star_weapon`).
* **Subsequent Cycles (Every 8 Issues)**: The cycle follows continuously on every 8 issues afterwards, alternating between a Weapon Selection Box and a Rate-up 6★ weapon. Effectively, this guarantees a direct `up_6_star_weapon` reward at the **18, 34, 50, 66, and 82** cumulative issue marks, serving as an absolute hard-pity anchor for worst-case scenarios.

**🔗 Sub-product Output & Yields**
* **Tracked Metrics**: Outputs total cumulative pulls (`num_pulls_for_weapon`), `up_6_star_weapon`, `non_up_6_star_weapon`, `5_star_weapon`, and `4_star_weapon`.
* **Token Yields (`blue_quota`)**: Dynamically tallies cumulative `blue_quota` generated across the full journey to Full Potential (6★ yields `50`, 5★ yields `10`).

[⬆️ Back to Top](./simulation.ipynb#sim_top)

---

**🇹🇼 繁體中文版**
***

**🔄 核心邏輯與環境設定**
本模式之抽取操作單位與常規機率設定，完全沿用 **[4. Arsenal Issues - First Acquisition of Rate-up Weapon / 武器池模擬—首次獲取機率提升武器](./simulation.ipynb#sim_mode_4)**。每次申領以一組十連抽（Issue）為運算單位，消耗 `1980 武庫配額`，並嚴格執行基礎掉落機率（六星：`4.0000%`、五星：`15.0000%`、四星：`81.0000%`）。

**🎯 終止條件**
* **終止判定規則**：本模式以達成武器「滿潛」為唯一目標。當模擬器統計到玩家所獲得的機率提升六星武器總數達到 6 把或以上時（`up_6_star_weapon >= 6`），模擬即刻終止。

**🛡️ 保底與避非機制（Fail-safe）**
* **單次申領五星或以上保底**：若前 9 抽皆未獲得 5★ 或 6★ 武器，則第 10 抽強制觸發保底，確保該抽結果至少為五星（5★ at minimum）。
* **連續 3 次申領六星保底**：連續 3 次申領皆未獲得六星武器時，第 4 次申領必定保底獲得六星武器。抽中任何六星武器時重置。
* **連續 7 次申領機率提升大保底（僅限一次）**：若連續 7 次申領皆未獲得機率提升六星武器，則第 8 次申領必定獲得 `up_6_star_weapon`。
  * > ⚠️ **運作行為注意**：此項大保底機制**在整個抽卡生命週期中僅生效一次**。一旦玩家獲取了第一把 `up_6_star_weapon`（不論是運氣好提前抽到，還是透過第 8 次大保底觸發），該避非機制就會失效，後續的滿潛進程將不再享有此 7 連非的大保底保護。
* **保底優先度控制**：若同時觸發五星最低保底與六星保底，系統判定上六星優先度高於五星。

**🎁 累計申領里程碑節點（滿潛核心推進器）**
由於達成滿潛需要較長的抽取縱深，玩家必定會觸發並循環里程碑機制。這些節點在長線模擬中提供了強力的期望值定錨：
* **累計 10 次申領**：獲得「非機率提升武器自選箱」，直接計入 `non_up_6_star_weapon`。
* **累計 18 次申領**：改為直接獲得一把當期機率提升六星武器（`up_6_star_weapon`）。
* **後續循環（每 8 次交替）**：在此之後，每申領 8 次即可額外交替獲得自選箱與機率提升武器。意即在累積申領達 **18、34、50、66、82** 次等里程碑節點時，程式碼會直接將 `up_6_star_weapon` 計入統計，成為最壞情況下的絕對硬保底。

**🔗 副產物配額與代幣統計**
* **統計欄位**：輸出達成滿潛時的總抽取次數 (`num_pulls_for_weapon`)、`up_6_star_weapon`、`non_up_6_star_weapon`、`5_star_weapon` 與 `4_star_weapon`。
* **代幣產出 (`blue_quota`)**：動態統計長途抽取過程中累積獲得的「集成配額（AIC Quota）」（六星產出 `50`，五星產出 `10`）。

---

[⬆️ 返回頂端 / Back to Top](./simulation.ipynb#sim_top)

In [6]:
class get_Full_Potential_UPweapon:
    def __init__(self):
        self.PROB_6_STAR = 0.04
        self.PROB_5_STAR = 0.15
        
        # State counters
        self.num_pulls_for_weapon = 0       # Total number of "multi-pulls" consumed (1, 2, 3...)
        self.pity_6_star_singles = 0        # CRITICAL: Relative single pulls elapsed since the last 6-star
        
        self.is_8th_multi_pity_active = True 
        self.blue_quota = 0
        
        self.inventory = {
            'up_6_star_weapon': 0,
            'non_up_6_star_weapon': 0,
            '5_star_weapon': 0,
            '4_star_weapon': 0
        }

    def _should_terminate(self):
        # Full Potential Graduation Criterion (up_6_star_weapon >= 6)
        return self.inventory['up_6_star_weapon'] >= 6

    def _roll_six_star(self, is_forced_up=False):
        """Helper to process a 6-star acquisition from pulls (Awards 50 blue_quota)"""
        self.blue_quota += 50  
        # Whichever way a 6-star is obtained, the relative counter resets to 0 immediately!
        self.pity_6_star_singles = 0 
        
        is_up = is_forced_up or (np.random.random() < 0.25)
        if is_up:
            self.inventory['up_6_star_weapon'] += 1
            self.is_8th_multi_pity_active = False 
        else:
            self.inventory['non_up_6_star_weapon'] += 1

    def execute_one_multi_pull(self):
        """Execute exactly ONE multi-pull batch (10 slots) with real-time rolling counters"""
        self.num_pulls_for_weapon += 1
        
        # Local trackers to monitor drops within this specific multi-pull batch
        six_stars_in_this_multi = 0
        five_stars_in_this_multi = 0
        
        # Process a 10-slot loop where each slot represents a real-time single pull sequence
        for slot in range(10):
            is_last_slot = (slot == 9)
            
            # --- Establish Real-Time Pity Conditions ---
            # 1. Big Pity: Absolute 80th pull (Last slot of 8th multi)
            force_up_6 = is_last_slot and self.is_8th_multi_pity_active and (self.num_pulls_for_weapon == 8)
            
            # 2. Regular Pity: Real-time 40th rolling single pull without any 6-star weapon
            force_any_6 = (self.pity_6_star_singles + 1 == 40)
            
            # Status flag to check if the current single pull yields a 6-star
            slot_yielded_6_star = False
            
            # --- Core Priority Interception (Evaluated dynamically per single pull) ---
            if force_up_6:
                self._roll_six_star(is_forced_up=True)
                six_stars_in_this_multi += 1
                slot_yielded_6_star = True
                
            elif force_any_6:
                # Real-time regular pity intercept: Triggers immediately at the exact 40th pull
                self._roll_six_star(is_forced_up=False)
                six_stars_in_this_multi += 1
                slot_yielded_6_star = True
                
            else:
                # Standard Probability Roll for the current slot
                rand = np.random.random()
                
                # A: Check if 6-star is rolled naturally (4% chance)
                if rand < self.PROB_6_STAR:
                    self._roll_six_star(is_forced_up=False)
                    six_stars_in_this_multi += 1
                    slot_yielded_6_star = True
                
                # B: If 6-star is NOT rolled, check if 5-star is rolled naturally OR forced by 10-pull pity
                # Pity condition: Last slot AND no 5-star or 6-star was found in the previous 9 slots
                elif (rand < (self.PROB_6_STAR + self.PROB_5_STAR)) or \
                     (is_last_slot and (six_stars_in_this_multi == 0) and (five_stars_in_this_multi == 0)):
                    self.inventory['5_star_weapon'] += 1
                    self.blue_quota += 10
                    five_stars_in_this_multi += 1
                
                # C: Otherwise, it's a 4-star weapon
                else:
                    self.inventory['4_star_weapon'] += 1
            
            # --- Real-Time Counter Maintenance ---
            # Increment the rolling counter only if NO 6-star weapon was obtained in this specific single pull.
            if not slot_yielded_6_star:
                self.pity_6_star_singles += 1

    def run(self):
        while not self._should_terminate():
            self.execute_one_multi_pull()
            
            current_count = self.num_pulls_for_weapon
            
            # Pattern A: Non-up Choice Box at 10, 26, 42, 58, 74
            if current_count == 10 or (current_count > 10 and (current_count - 10) % 16 == 0):
                self.inventory['non_up_6_star_weapon'] += 1
                
            # Pattern B: Up Weapon Milestone at 18, 34, 50, 66, 82
            if current_count == 18 or (current_count > 18 and (current_count - 18) % 16 == 0):
                self.inventory['up_6_star_weapon'] += 1
                
            if self._should_terminate():
                break
            
        return {
            'num_pulls_for_weapon': self.num_pulls_for_weapon,
            'up_6_star_weapon': self.inventory['up_6_star_weapon'],
            'non_up_6_star_weapon': self.inventory['non_up_6_star_weapon'],
            '5_star_weapon': self.inventory['5_star_weapon'],
            '4_star_weapon': self.inventory['4_star_weapon'],
            'blue_quota': self.blue_quota
        }

# --- Execute Simulations Pipeline ---
def main_process(total_sims):
    results = []
    print(
        f"Generating {total_sims} simulation records for Full Potential UP Weapon..."
    )

    # 💡 String-based path construction using standard os.path
    save_path = os.path.join(DATA_DIR, "sim_weapon_full_potential.csv")

    # 💡 Core Enhancement: Safe pre-execution file purging via os module
    # If a residual leftover file exists, delete it individually to guarantee data purity
    if os.path.exists(save_path):
        os.remove(save_path)

    # Prevent ZeroDivisionError when total_sims is set below 10 for rapid validation
    progress_step = max(1, total_sims // 10)

    for i in range(total_sims):
        sim = get_Full_Potential_UPweapon()
        results.append(sim.run())

        # Progress report evaluated with guarded step length
        if (i + 1) % progress_step == 0:
            print(f"Progress: {(i + 1) / total_sims * 100:.0f}%")

    df = pd.DataFrame(results)

    # Save to CSV
    df.to_csv(save_path, index=False)
    print(f"Done! Raw data safely exported and saved to: {save_path}\n")

if __name__ == "__main__":
    main_process(trials)

Generating 1000 simulation records for Full Potential UP Weapon...
Progress: 10%
Progress: 20%
Progress: 30%
Progress: 40%
Progress: 50%
Progress: 60%
Progress: 70%
Progress: 80%
Progress: 90%
Progress: 100%
Done! Raw data safely exported and saved to: sim_data\sim_weapon_full_potential.csv

